# 01 — Data Audit & Model (Allegheny County Pilot)

**Independent portfolio case study — Allegheny County, PA (Pittsburgh area)**

Target: recorded `sale_price` (USD) — not assessed value. Market is one county, time-boxed 2019–2026, WPRDC snapshot 2026-10-30 (CC0). This notebook is end-to-end runnable with the shipped `data/sample_properties.csv`; to reproduce from the full CC0 download, follow `data/README.md`.

Steps: (1) audit candidate sources, (2) filter & provenance, (3) EDA, (4) temporal split & pipelines, (5) Ridge vs XGBoost with holdout RMSE/MAE/R² + 80% interval, (6) SHAP global + local.


## 0) Setup

In [ ]:
import pathlib, json, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path().resolve().parent if Path().resolve().name=='notebooks' else Path('.')
# when run from notebooks/, ROOT is repo root; when run from repo root, it's '.'
import sys
sys.path.append(str(ROOT))
from src.data_loader import load_sample
from src.config import PRICE_MIN, PRICE_MAX, TARGET_COL
pd.set_option('display.max_columns', 40)
print('ROOT', ROOT.resolve())

## 1) Data audit summary (see docs/data_audit.md for full table)

We audited 4 families: Allegheny WPRDC (CC0, chosen), King County WA (public but no explicit CC0, 3-table join), Zillow/Redfin (aggregated indices only, no parcel target), Ames/Kaggle (1,460-row benchmark, not a market). The choice is Allegheny because it has row-level sale price + 16 building features in one CC0 table with 147k valid sales and weekly updates.


In [ ]:
audit = pd.DataFrame([
  ['Allegheny Assessments (WPRDC)', 'sale_price (recorded)', 'Allegheny County, 2013–present', '147,581 valid', 'CC0 — chosen'],
  ['Allegheny Sales (WPRDC)', 'PRICE (recorded)', 'same', '~140k valid', 'CC0 — check'],
  ['King County WA', 'sale price (excise)', 'King County, 2004–present', '~25k/yr', 'public, no explicit CC0'],
  ['Zillow/Redfin', 'ZHVI/median (aggregated)', 'USA/ZIP', 'n/a parcel', 'free with attribution — context only'],
  ['Ames/Kaggle', 'SalePrice', 'Ames, IA 2006–10', '2,919', 'competition — benchmark only'],
], columns=['Source','Target','Geo/time','Usable rows','Licence'])
audit

## 2) Load curated sample & provenance

In [ ]:
df = load_sample()
print(f"Loaded {len(df)} rows, {len(df.columns)} cols")
df.head(3).T


In [ ]:
# Provenance
print('Snapshot: 2026-10-30 (WPRDC CC0)')
print('Coverage:', df['sale_date'].min(), '→', df['sale_date'].max())
print('Sale code unique:', df['sale_code'].unique())
print('Price median:', df['sale_price'].median(), 'mean', round(df['sale_price'].mean()))
print('ZIPs:', df['zip_code'].nunique(), 'Municipalities:', df['municipality'].nunique())
df.describe(include='all').head()


## 3) EDA — price vs living area, ZIP premiums, condition

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.scatterplot(data=df.sample(800), x='living_area_sqft', y='sale_price', hue='condition', alpha=0.5, ax=axes[0])
axes[0].set_title('Price vs Living Area (sample 800)')
sns.histplot(df['sale_price'], bins=50, ax=axes[1], color='#2C4A5A')
axes[1].set_title('Sale price distribution')
plt.tight_layout(); plt.savefig(ROOT/'models'/'eda_price.png', dpi=150, bbox_inches='tight'); plt.show()
zip_med = df.groupby('zip_code')['sale_price'].median().sort_values(ascending=False).head(10)
print(zip_med)
df.groupby('condition')['sale_price'].median().sort_values()


## 4) Filtering & temporal split (leak-safe)

We keep PRICE $25k–$2M, living 300–8,000 sq ft, `FINISHEDLIVINGAREA` present, `SALECODE=0`, residential. Excluded assessed values from features to avoid leakage. Split is temporal (oldest 80% train, newest 20% holdout) — see docs/assumptions_and_limits.md for drift note.


In [ ]:
from src.model import train_all
results, models, extra = train_all(df)
import json
print(json.dumps(results, indent=2))
# Save already done inside train_all

## 5) Metrics & diagnostics

In [ ]:
from src.evaluation import price_band_analysis, residual_plot
X_test, y_test, y_pred, residuals = extra
band = price_band_analysis(y_test.values, y_pred, y_test.values)
band


In [ ]:
residual_plot(y_test.values, y_pred, save_path=ROOT/'models'/'residual_plot.png')
plt.show()
print('Mean residual (bias):', residuals.mean(), '=> recent holdout underpredicted due to appreciation')
print('Abs80:', np.percentile(np.abs(residuals),80))


## 6) SHAP — global & local


In [ ]:
from src.shap_explainer import compute_shap, plot_shap_bar
from src.model import temporal_train_test_split
tr, te = temporal_train_test_split(df, 'sale_date', 0.2)
cols = ['zip_code','municipality','living_area_sqft','lot_area_sqft','bedrooms','full_baths','half_baths','total_rooms','year_built','age_at_sale','condition','grade','exterior_finish','stories','has_basement_garage','fireplaces']
X_train, X_test = tr[cols], te[cols]
best = models['xgb']
shap_vals, feat_names, importance, explainer = compute_shap(best, X_train.sample(500, random_state=42), X_test.sample(300, random_state=42))
if importance is not None:
    importance.head(12)


In [ ]:
if importance is not None:
    plot_shap_bar(importance, save_path=ROOT/'models'/'shap_bar.png')
    plt.show()


### Local example
Pick one holdout home and show top 8 SHAP contributions.


In [ ]:
if shap_vals is not None:
    import pandas as pd
    df_local = pd.DataFrame({'feature': feat_names, 'shap': shap_vals[0]})
    df_local['abs'] = df_local['shap'].abs()
    df_local.sort_values('abs', ascending=False).head(8)


## 7) What’s next

- Rent/deposit/photo are Roadmap only (require separate jurisdiction-safe sources).
- To reproduce from full CC0 download, see `data/README.md` → download `assessments.csv` → re-run this notebook (it will regenerate the sample and models).
- App is `app.py` — `streamlit run app.py`.
